# 🧪 Stage 1: Pretraining (Unconditional Model)Trains `VanillaMolGen_RNN` on ChEMBL SMILES data.**GPU Utilization:** Uses all available GPUs via MXNet data parallelism.**Expected time:** ~6-12 hours for 480K iterations on 2× L40 GPUs.

In [ ]:
import os, sysCTDDG_ROOT = os.environ.get("CTDDG_ROOT", os.path.dirname(os.getcwd()))os.environ["CTDDG_ROOT"] = CTDDG_ROOTos.environ["MXNET_CUDNN_LIB_CHECKING"] = "0"os.chdir(CTDDG_ROOT)print(f"Working directory: {os.getcwd()}")

## GPU Check

In [ ]:
import mxnet as mxNUM_GPUS = mx.context.num_gpus()print(f"Available GPUs: {NUM_GPUS}")!nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader

## Run PretrainingThis executes the original `pretraining.ipynb` notebook.The training loop uses 480,000 iterations with checkpointing every 500 steps.

In [ ]:
%%time# Execute the original pretraining notebook# This runs the entire single-cell notebook which contains all model definitions + training loopimport subprocess, sysresult = subprocess.run(    [sys.executable, "-m", "jupyter", "nbconvert",     "--to", "notebook", "--execute",     "--ExecutePreprocessor.timeout=172800",  # 48h timeout     "--ExecutePreprocessor.kernel_name=ctddg_env",     f"--output=pretraining_executed.ipynb",     os.path.join(CTDDG_ROOT, "code", "pretraining.ipynb")],    capture_output=True, text=True, cwd=CTDDG_ROOT)print("STDOUT:", result.stdout[-2000:] if result.stdout else "")if result.returncode != 0:    print("STDERR:", result.stderr[-3000:])    print(f"\n❌ Pretraining failed (exit code {result.returncode})")else:    print("\n✅ Pretraining completed successfully!")

## Monitor Training ProgressRun this cell periodically to check training progress.

In [ ]:
log_path = os.path.join(CTDDG_ROOT, "outputs", "pretrain", "logs", "log.out")if os.path.exists(log_path):    with open(log_path) as f:        lines = f.readlines()    print(f"Log entries: {len(lines)}")    if len(lines) > 1:        print(f"Header: {lines[0].strip()}")        print(f"Latest: {lines[-1].strip()}")        if lines[-1].strip() == "Training finished":            print("\n🎉 Training is COMPLETE!")        else:            parts = lines[-1].strip().split('\t')            if len(parts) >= 2:                print(f"\nProgress: step {parts[0]} / 480000 ({100*int(parts[0])/480000:.1f}%)")else:    print("⏳ Training has not started yet (no log file)")